In [1]:
import pandas as pd
df = pd.read_csv("Mammographic.csv")

In [2]:
print(df.dtypes)

Unnamed: 0    int64
BI-RADS       int64
Age           int64
Shape         int64
Margin        int64
Density       int64
Severity      int64
dtype: object


In [3]:

df['BI-RADS'] = df['BI-RADS'].astype(str)
df['Shape'] = df['Shape'].astype(str)
df['Margin'] = df['Margin'].astype(str)
df['Density'] = df['Density'].astype(str)
print(df.dtypes)


Unnamed: 0     int64
BI-RADS       object
Age            int64
Shape         object
Margin        object
Density       object
Severity       int64
dtype: object


In [4]:
df = df.drop('Unnamed: 0', axis=1)

In [5]:
print(df.dtypes)

BI-RADS     object
Age          int64
Shape       object
Margin      object
Density     object
Severity     int64
dtype: object


In [6]:
print(df.dtypes)

BI-RADS     object
Age          int64
Shape       object
Margin      object
Density     object
Severity     int64
dtype: object


In [7]:
from typing import List, Any
import numpy as np

def clean_dataset(
    df: pd.DataFrame,
    cols_to_drop: List[Any] = None,
    missing_values: List[Any] = None
) -> pd.DataFrame:
    """
    Предобработка датафрейма с учетом пользовательских меток пропусков:
    1. Заполняет пропущенные значения
       - Модой для категориальных/текстовых признаков.
       - Средним значением для числовых признаков.
    2. Удаляет признаки, у которых одно значение занимает 100% строк.
    """
    df_cleaned = df.copy()
    n_rows = len(df_cleaned)
    
    if n_rows == 0:
        return df_cleaned

    # Задаем список меток для пропущенных значений по умолчанию, если не передан свой
    if missing_values is None:
        missing_values = ['', ' ', 'N/A', 'NA', 'null', 'nan', 'NaN', 'None', 'none', '?', '-']

    # 1. Приведение пользовательских пропусков к np.nan и их импутация
    for col in df_cleaned.columns:
        # Заменяем все кастомные метки пропусков на np.nan
        df_cleaned[col] = df_cleaned[col].replace(missing_values, np.nan)

        if df_cleaned[col].isna().sum() > 0:
            if pd.api.types.is_numeric_dtype(df_cleaned[col]):
                # а) Числовые переменные -> средним значением
                mean_val = df_cleaned[col].mean()
                df_cleaned[col] = df_cleaned[col].fillna(mean_val)
            else:
                # б) Категориальные переменные -> модой
                mode_val = df_cleaned[col].mode()
                if not mode_val.empty:
                    df_cleaned[col] = df_cleaned[col].fillna(mode_val[0])

    # 2. Проверка условий для удаления колонок
    for col in df_cleaned.columns:
        series = df_cleaned[col]
        max_value_count = series.value_counts().iloc[0]
        max_value_ratio = max_value_count / n_rows

        # Если одинаковых значений = 100% -> к удалению
        if max_value_ratio == 1:
            cols_to_drop.append(col)

    # Удаляем выявленные колонки
    if cols_to_drop:
        df_cleaned = df_cleaned.drop(columns=cols_to_drop)
    return df_cleaned

df = clean_dataset(df)

In [8]:
print(df.dtypes)

BI-RADS     object
Age          int64
Shape       object
Margin      object
Density     object
Severity     int64
dtype: object


In [9]:
y = df["Severity"]
X = df.drop(["Severity"], axis=1)

In [10]:
import time
import category_encoders as ce
from skrub import SimilarityEncoder, MinHashEncoder
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate, RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC

from k_fold_encoder import KFoldTargetEncoder
from k_fold_glmm_encoder import KFoldGLMMEncoder

cat_cols = ['BI-RADS','Shape','Margin','Density']

encoders = {
    "MinHash": lambda: ColumnTransformer([(
                f"minhash_{col}",
                MinHashEncoder(),
                [col]
            )
            for col in cat_cols
        ],
        remainder="passthrough"
    ),
    "Similarity": lambda: ColumnTransformer([
        ("cat", SimilarityEncoder(), cat_cols)], 
        remainder="passthrough"),
   'Leave-one-out': lambda: ce.LeaveOneOutEncoder(cols=cat_cols),
   'K-fold':        lambda: KFoldTargetEncoder(cols=cat_cols),
   "CatBoost":      lambda: ce.CatBoostEncoder(cols=cat_cols),
   "CV-GLMM":       lambda: KFoldGLMMEncoder(cols=cat_cols),
   "WOE":           lambda: ce.WOEEncoder(cols=cat_cols),
   "JS":            lambda: ce.JamesSteinEncoder(cols=cat_cols),
   "MT":            lambda: ce.TargetEncoder(cols=cat_cols),
   'GLMM':          lambda: ce.GLMMEncoder(cols=cat_cols),
   'M-Estimate':    lambda: ce.MEstimateEncoder(cols=cat_cols),
   "OneHot":  lambda: ce.OneHotEncoder(cols=cat_cols),
   "Gray":    lambda: ce.GrayEncoder(cols=cat_cols),
   "BaseN":   lambda: ce.BaseNEncoder(cols=cat_cols),
   "Hashing": lambda: ce.HashingEncoder(cols=cat_cols,n_components=32),
   "Helmert": lambda: ce.HelmertEncoder(cols=cat_cols),
   "Sum":     lambda: ce.SumEncoder(cols=cat_cols),
   "BackwardDifference": lambda: ce.BackwardDifferenceEncoder(cols=cat_cols)
}

models = {
    "LogReg": LogisticRegression(max_iter=1000),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), activation="identity", max_iter=1000, random_state=42),
    "LinearSVM": LinearSVC(dual="auto", random_state=42),
    "CART": DecisionTreeClassifier(random_state=42),
    "RF": RandomForestClassifier(random_state=42, n_jobs=4),
    "XGBoost": XGBClassifier(eval_metric="logloss", n_jobs=4),
    "LightGBM": LGBMClassifier(verbose=-1, n_jobs=4),
    "CatBoost": CatBoostClassifier(verbose=0, thread_count=4)
}

records = []

# Инициализируем генератор разбиений
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

for model_name, model_template in models.items():
    for encoder_name, encoder in encoders.items():
        
        split_idx = 0
        # Проходим по всем фолдам вручную
        for train_idx, test_idx in cv.split(X, y):
            start_time = time.time()

            # 1. Разбиение выборки на train и test
            X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

            # 2. Инициализация и обработка кодировщиком
            enc_instance = encoder()  # Вызываем lambda, получая объект кодировщика
            X_train_encoded = enc_instance.fit_transform(X_train, y_train)
            X_test_encoded = enc_instance.transform(X_test)

            # 3. Масштабирование (если требуется для модели)
            if model_name in ["LogReg", "LinearSVM", "MLP"]:
                scaler = StandardScaler()
                X_train_processed = scaler.fit_transform(X_train_encoded)
                X_test_processed = scaler.transform(X_test_encoded)
            else:
                X_train_processed = X_train_encoded
                X_test_processed = X_test_encoded

            # 4. Обучение модели (клонируем или создаем заново, чтобы сбросить состояние)
            model = clone(model_template)
            model.fit(X_train_processed, y_train)

            # 5. Получение предсказаний
            y_pred = model.predict(X_test_processed)

            # Для ROC AUC и PR AUC нужны вероятности или decision_function
            if hasattr(model, "predict_proba"):
                y_proba = model.predict_proba(X_test_processed)[:, 1]
            elif hasattr(model, "decision_function"):
                y_proba = model.decision_function(X_test_processed)
            else:
                y_proba = y_pred

            # 6. Расчет метрик качество (scoring)
            auc_score = roc_auc_score(y_test, y_proba)
            pr_auc_score = average_precision_score(y_test, y_proba)
            acc_score = accuracy_score(y_test, y_pred)
            f1_val = f1_score(y_test, y_pred)
            prec_val = precision_score(y_test, y_pred)
            rec_val = recall_score(y_test, y_pred)

            elapsed_time = time.time() - start_time

            # 7. Вычисление номера повтора (repeat) и фолда (fold)
            repeat = split_idx // 5 + 1
            fold = split_idx % 5 + 1

            records.append(
                {
                    "approach": "Empirical",
                    "dataset": "Mammographic",
                    "model": model_name,
                    "encoder": encoder_name,
                    "repeat": repeat,
                    "fold": fold,
                    "auc": auc_score,
                    "pr_auc": pr_auc_score,
                    "accuracy": acc_score,
                    "f1": f1_val,
                    "precision": prec_val,
                    "recall": rec_val,
                    "total_time ": elapsed_time,
                }
            )

            print(
                f"Empirical, Mammographic, {model_name}, {encoder_name}, {repeat}, {fold}, "
                f"{auc_score}, {pr_auc_score}, {acc_score}, {f1_val}, "
                f"{prec_val}, {rec_val}, {elapsed_time}"
            )

            split_idx += 1
results = pd.DataFrame(records)
results.to_csv("results_mammographic_empirical.csv", index=True)
print("Done!")

Empirical, Mammographic, LogReg, MinHash, 1, 1, 0.9240377632534494, 0.9342730606495105, 0.8734939759036144, 0.8662420382165605, 0.8947368421052632, 0.8395061728395061, 0.06312704086303711
Empirical, Mammographic, LogReg, MinHash, 1, 2, 0.9127814088598402, 0.9074556804793589, 0.8433734939759037, 0.8395061728395061, 0.8395061728395061, 0.8395061728395061, 0.04814314842224121
Empirical, Mammographic, LogReg, MinHash, 1, 3, 0.9185911401597674, 0.8986171936166933, 0.8373493975903614, 0.8163265306122449, 0.9090909090909091, 0.7407407407407407, 0.042231082916259766
Empirical, Mammographic, LogReg, MinHash, 1, 4, 0.8623546511627908, 0.8057982499695264, 0.7951807228915663, 0.7901234567901234, 0.7804878048780488, 0.8, 0.042649269104003906
Empirical, Mammographic, LogReg, MinHash, 1, 5, 0.9034156976744185, 0.8992800868867502, 0.8253012048192772, 0.8220858895705522, 0.8072289156626506, 0.8375, 0.043150901794433594
Empirical, Mammographic, LogReg, MinHash, 2, 1, 0.8928830791575889, 0.84373972600532

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, Mammographic, CART, Leave-one-out, 3, 2, 0.48264342774146696, 0.48439684664584265, 0.4939759036144578, 0.023255813953488372, 0.2, 0.012345679012345678, 0.03712773323059082
Empirical, Mammographic, CART, Leave-one-out, 3, 3, 0.5, 0.4879518072289157, 0.5120481927710844, 0.0, 0.0, 0.0, 0.04430508613586426
Empirical, Mammographic, CART, Leave-one-out, 3, 4, 0.4838662790697675, 0.476355421686747, 0.5, 0.06741573033707865, 0.3333333333333333, 0.0375, 0.03696465492248535
Empirical, Mammographic, CART, Leave-one-out, 3, 5, 0.5148255813953488, 0.4894558365913987, 0.5, 0.6406926406926406, 0.4900662251655629, 0.925, 0.03745317459106445
Empirical, Mammographic, CART, Leave-one-out, 4, 1, 0.7862745098039216, 0.7432957636999611, 0.7891566265060241, 0.7552447552447552, 0.8709677419354839, 0.6666666666666666, 0.036329030990600586
Empirical, Mammographic, CART, Leave-one-out, 4, 2, 0.5, 0.4879518072289157, 0.5120481927710844, 0.0, 0.0, 0.0, 0.036264896392822266


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricW

Empirical, Mammographic, CART, Leave-one-out, 4, 3, 0.5, 0.4879518072289157, 0.5120481927710844, 0.0, 0.0, 0.0, 0.04410696029663086
Empirical, Mammographic, CART, Leave-one-out, 4, 4, 0.49462209302325577, 0.480070281124498, 0.5120481927710844, 0.024096385542168676, 0.3333333333333333, 0.0125, 0.038156986236572266
Empirical, Mammographic, CART, Leave-one-out, 4, 5, 0.5, 0.4819277108433735, 0.5180722891566265, 0.0, 0.0, 0.0, 0.035308837890625
Empirical, Mammographic, CART, Leave-one-out, 5, 1, 0.477923021060276, 0.47987718068039353, 0.4879518072289157, 0.10526315789473684, 0.35714285714285715, 0.06172839506172839, 0.035314321517944336
Empirical, Mammographic, CART, Leave-one-out, 5, 2, 0.5, 0.4879518072289157, 0.5120481927710844, 0.0, 0.0, 0.0, 0.03538990020751953
Empirical, Mammographic, CART, Leave-one-out, 5, 3, 0.4944081336238199, 0.48604293718082203, 0.5060240963855421, 0.023809523809523808, 0.3333333333333333, 0.012345679012345678, 0.039758920669555664
Empirical, Mammographic, CART

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


... (71 outputs left out)

C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, Mammographic, CatBoost, Leave-one-out, 5, 2, 0.8692810457516339, 0.8352326210716434, 0.5120481927710844, 0.0, 0.0, 0.0, 0.9293539524078369
Empirical, Mammographic, CatBoost, Leave-one-out, 5, 3, 0.892955700798838, 0.8241252998788543, 0.5060240963855421, 0.023809523809523808, 0.3333333333333333, 0.012345679012345678, 0.8748433589935303
Empirical, Mammographic, CatBoost, Leave-one-out, 5, 4, 0.8359738372093023, 0.7644518903871307, 0.7650602409638554, 0.7577639751552795, 0.7530864197530864, 0.7625, 0.8740522861480713


C:\Anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Empirical, Mammographic, CatBoost, Leave-one-out, 5, 5, 0.8604651162790697, 0.8150021212965723, 0.5180722891566265, 0.0, 0.0, 0.0, 1.027651309967041
Empirical, Mammographic, CatBoost, K-fold, 1, 1, 0.9286855482933914, 0.9295781411526145, 0.8795180722891566, 0.8765432098765432, 0.8765432098765432, 0.8765432098765432, 1.0918357372283936
Empirical, Mammographic, CatBoost, K-fold, 1, 2, 0.8847494553376906, 0.8548416350569987, 0.8313253012048193, 0.825, 0.8354430379746836, 0.8148148148148148, 0.921776294708252
Empirical, Mammographic, CatBoost, K-fold, 1, 3, 0.9143790849673202, 0.8824567093816457, 0.8313253012048193, 0.8055555555555556, 0.9206349206349206, 0.7160493827160493, 0.9085793495178223
Empirical, Mammographic, CatBoost, K-fold, 1, 4, 0.8536337209302325, 0.801234874617383, 0.7951807228915663, 0.7976190476190477, 0.7613636363636364, 0.8375, 0.8958733081817627
Empirical, Mammographic, CatBoost, K-fold, 1, 5, 0.8803052325581395, 0.8685448213215727, 0.7831325301204819, 0.769230769230769